<a href="https://colab.research.google.com/github/BG-bibek/BG-bibek/blob/main/Bg_Binary_Classification_in_Tensorflow_with_Deep_Learning_refined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instructions for students**

1. Make your **own copy** of this notebook before editing (so your work is saved).
2. Run cells **top to bottom**. If you get errors, restart the runtime/kernel and run again.
3. As you work, answer the **“Quick check”** questions in markdown cells (write your answers under them).
4. Optional: try the **Extension** sections if you finish early.

---

**Learning objectives (by the end you should be able to):**
- Explain what **binary classification** is and when to use it.
- Prepare text data for a neural network (vectorization, train/validation/test splits).
- Build and train a simple TensorFlow/Keras model for classification.
- Diagnose **overfitting** using learning curves.
- Evaluate a classifier beyond accuracy (confusion matrix, precision/recall, ROC-AUC).
- Perform basic error analysis and threshold tuning.

**Prerequisites:** Python basics, NumPy arrays, and a rough idea of what a neural network is.


# Binary Text Classification in TensorFlow (IMDB Sentiment)

## Classifying movie reviews (positive vs. negative)

In this notebook you’ll train a neural network to predict whether a movie review is **positive (1)** or **negative (0)**.

Why this matters:
- Many real problems are binary: **spam vs. not spam**, **fraud vs. legit**, **disease vs. healthy**, **churn vs. stay**.
- The workflow (data → model → training → evaluation → iteration) generalizes to other ML tasks.

> **Quick check:** What is one real-world problem you can phrase as a binary classification task?


### The IMDB dataset

**Loading the IMDB dataset**

We’ll use the built-in Keras IMDB dataset: 50,000 movie reviews, already converted to sequences of integer word IDs.
- 25,000 for training
- 25,000 for testing

We’ll also limit the vocabulary to the **top 10,000 most frequent words** to keep the input size manageable.


The IMDB dataset contains 50,000 reviews from the Internet Movie Database (imdb.com).
By default, the data is split into a **training** set and a **test** set, each with 25,000 reviews.

Each review is stored as a list of integers (word indices). The label is:
- `1` = positive review
- `0` = negative review


In [ ]:
from tensorflow.keras.datasets import imdb
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(
    num_words=10000)

Lets inspect the data.

Executing the cell below, we see that the total number of records in the training dataset is 25000 rows.

We can also see that the first movie review at index 0 is a sentence containing 218 words.

The actual data in the first record (at index 0) contains a sentence where each word has been converted to an integer. This integer corresponds to a word in a word-dictionary.

In [ ]:
print(len(train_data))
print(len(train_data[0]))
print(train_data[0])

Let’s inspect the labels. The corresponding label for the review we viewed above has a value of **1**.

> **Quick check:** What does a label of `0` mean in this dataset?


In [ ]:
train_labels[0]

The cell below shows that the length of the review with the most words is 9999.

In [ ]:
max([max(sequence) for sequence in train_data])

**Decoding reviews back to text**

Convert the word indices of the review at index 0 of the training dataset back to a string sentence. Now we can read the review. We will replace missing words with a question mark. Remember we only kept the most frequently used words and discarded the rest.

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])
decoded_review = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in train_data[0]])
decoded_review

### Preparing the data

**Encoding the integer sequences (multi-hot / bag-of-words)**

Neural networks expect fixed-size numeric inputs. Reviews have different lengths, so we’ll convert each review (a list of word IDs)
into a **10,000-dimensional vector**:

- Position *j* is `1` if word *j* appears at least once in the review, else `0`.
- This is similar to a **bag-of-words** representation (presence/absence, not counts).

✅ Pros: simple, fast baseline.  
⚠️ Cons: ignores word order (e.g., “not good” vs “good”).

Later, in the extension tasks, you’ll try an **Embedding** model that preserves some order/sequence information.


In [ ]:
import numpy as np
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        for j in sequence:
            results[i, j] = 1.
    return results
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

We store these in new variables x_train and x_test.
In the cell below, we see that the review at position 0 has now been multi-hot encoded. Compare this with the same review in the old variable train_data[0] from above.

In [ ]:
print(x_train[0])

x_train refers to our training data, y_train contains the labels for it.
Similarly, x_test refers to our testing data and y_test containes the labels for it.

In [ ]:
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

In [ ]:
y_train[0]

### Building your model

**Model definition (baseline MLP)**

Our inputs are 10,000-dimensional vectors and our labels are scalars (0/1).
We’ll start with a simple **multi-layer perceptron (MLP)**:

- Dense layer(s) with **ReLU**: learn non-linear patterns.
- Final Dense layer with **Sigmoid**: outputs a probability `p(positive)` between 0 and 1.

> **Quick check:** Why do we use *sigmoid* (not softmax) for binary classification?


**ReLU activation**

ReLU (Rectified Linear Unit) is widely used because it is simple and helps gradients flow:
- `ReLU(x) = max(0, x)`

It keeps positive values and clips negatives to zero, adding non-linearity.


**Sigmoid activation**

For binary classification, sigmoid maps a real number to a probability:
- Output in (0, 1)
- Interpretable as `p(y=1 | x)`

During evaluation, we often convert probabilities to class labels using a **threshold** (commonly 0.5).


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

tf.random.set_seed(1234)
np.random.seed(1234)

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

**Compiling the model**

We choose:
- **Optimizer:** `rmsprop` (a solid default for many problems)
- **Loss:** `binary_crossentropy` (standard for binary classification with sigmoid)
- **Metric:** `accuracy` (useful, but not always sufficient — we’ll add more metrics later)

> **Quick check:** When can accuracy be misleading?


In [ ]:
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

### Validating your approach

**Setting aside a validation set**

We split off a **validation** set from the training data. This is used during training to:
- monitor generalization
- detect overfitting
- tune hyperparameters (layer sizes, learning rate, etc.)

Important: we do **not** use the test set until the very end.


In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

In [ ]:
len(x_val)

**Training your model**

An **epoch** = one complete pass through the training set.  
We also choose a **batch size** (how many samples per gradient update).

We’ll train for 20 epochs so we can *see* overfitting in the curves.


In [ ]:
history = model.fit(partial_x_train,
                    partial_y_train,
                    epochs=20,
                    batch_size=512,
                    validation_data=(x_val, y_val))

The `history` object stores losses and metrics for training and validation at each epoch.
We’ll plot these to diagnose underfitting/overfitting.


In [ ]:
history_dict = history.history
history_dict.keys()

**Plotting training vs. validation loss**

- If training loss keeps going down but validation loss starts going up, the model is overfitting.
- The “best” epoch is typically around the minimum validation loss.

> **Quick check:** In your plot, around which epoch does overfitting begin?


In [ ]:
import matplotlib.pyplot as plt
history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

**Plotting training vs. validation accuracy**

Accuracy often continues improving even when validation loss worsens.  
That’s one reason we look at **multiple metrics**.


In [ ]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

**Retraining a model from scratch (a better choice of epochs)**

After looking at the curves, we retrain from scratch using a smaller number of epochs (e.g., 4),
then evaluate on the test set.

Tip: in real projects you can use **EarlyStopping** to stop training automatically (extension task).


In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

The results show **test loss** and **test accuracy**.

State-of-the-art models can reach higher accuracy on this dataset, but our goal here is to understand the full workflow,
not to win a leaderboard.


In [ ]:
results

### Using a trained model to generate predictions

`model.predict` returns **probabilities** (not class labels).
For example, a prediction of `0.91` means “91% chance of positive”.

To convert probabilities to class labels, we choose a **threshold**, often 0.5:
- `p >= 0.5` → predict positive
- `p < 0.5` → predict negative

We’ll also explore how changing the threshold affects precision/recall.


In [ ]:
predictions = model.predict(x_test)

In [ ]:
predictions[0:10]

In [ ]:
y_test[0:10]

Let’s decode a test review back to text so we can read it.

> **Quick check:** Do you agree with the label and the model’s prediction for this review? Why/why not?


In [ ]:
data = test_data
review_index_to_convert = 0

word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])
decoded_review = " ".join(
    [reverse_word_index.get(i - 3, "?") for i in data[review_index_to_convert]])
decoded_review

### Confusion Matrix (and more metrics)

A confusion matrix summarizes predictions vs. true labels:

- **TP** (true positive): predicted positive, actually positive  
- **TN** (true negative): predicted negative, actually negative  
- **FP** (false positive): predicted positive, actually negative  
- **FN** (false negative): predicted negative, actually positive  

From these we compute:
- **Accuracy** = (TP + TN) / all
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)
- **F1** = harmonic mean of precision & recall

We’ll compute the confusion matrix correctly using **class labels** (after thresholding).


In [ ]:
# Convert probabilities to class labels using a threshold
import numpy as np
import tensorflow as tf

proba = predictions.reshape(-1)  # (N,)
threshold = 0.5
y_pred = (proba >= threshold).astype(int)

confusion = tf.math.confusion_matrix(labels=y_test.astype(int), predictions=y_pred, num_classes=2)
confusion


In [ ]:
print(confusion.numpy())

In [ ]:
import matplotlib.pyplot as plt
from sklearn import metrics

cm = metrics.confusion_matrix(y_test.astype(int), y_pred)
disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["neg (0)", "pos (1)"])
disp.plot(values_format="d")
plt.title(f"Confusion matrix (threshold = {threshold})")
plt.show()

# Derived metrics
print(metrics.classification_report(y_test.astype(int), y_pred, digits=3))


In [ ]:
# ROC curve and AUC
fpr, tpr, roc_thresholds = metrics.roc_curve(y_test, proba)
roc_auc = metrics.auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()

# Precision-Recall curve
precision, recall, pr_thresholds = metrics.precision_recall_curve(y_test, proba)
pr_auc = metrics.auc(recall, precision)

plt.figure()
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.legend()
plt.show()


In [ ]:
# Threshold tuning: choose a threshold that maximizes F1 on the TEST set (for demo only)
# In real work, tune threshold on a validation set, not on the test set.
thresholds = np.linspace(0.05, 0.95, 19)
best = None
for th in thresholds:
    yp = (proba >= th).astype(int)
    f1 = metrics.f1_score(y_test.astype(int), yp)
    if best is None or f1 > best["f1"]:
        best = {"threshold": th, "f1": f1,
                "precision": metrics.precision_score(y_test.astype(int), yp),
                "recall": metrics.recall_score(y_test.astype(int), yp)}
best


Notice how precision and recall trade off as you change the threshold.

> **Quick check:** If a false negative is *more costly* than a false positive (e.g., missing a disease), should you move the threshold up or down?


## Challenging questions & extension tasks

Try these once you’ve completed the main notebook.

### Concept questions
1. **Order matters:** Why can bag-of-words struggle with phrases like “not good”? What representation could help?
2. **Overfitting:** What evidence of overfitting did you observe in your plots? List two ways to reduce it.
3. **Metrics:** Give an example of a real-world task where accuracy is a bad metric and explain what you would use instead.
4. **Thresholds:** If you care more about catching positives (high recall), what generally happens to precision?

### Hands-on tasks (recommended)
1. **Early stopping & checkpoints**
   - Add `EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)`
   - Add `ModelCheckpoint` to save the best model
   - Compare the final test performance.

2. **Regularization study**
   - Add `Dropout(0.5)` after the first Dense layer
   - Add `kernel_regularizer=tf.keras.regularizers.l2(1e-4)`
   - Compare learning curves and test metrics.

3. **Vocabulary size sweep**
   - Try `num_words` = 5,000 and 20,000
   - Plot how performance changes and discuss the trade-offs.

4. **Embedding model (sequence-based)**
   - Instead of multi-hot vectors, pad sequences and use:
     - `Embedding(num_words, 16)`
     - `GlobalAveragePooling1D()` or an `LSTM/GRU`
   - Compare against the bag-of-words baseline.

5. **Error analysis**
   - Find 10 **highest-confidence wrong predictions** (largest |p - y|)
   - Read the decoded reviews and categorize error types:
     - sarcasm/irony, negation, rare words, mixed sentiment, etc.

### Stretch (hard)
1. **Calibration**
   - Are predicted probabilities well calibrated?
   - Plot a reliability diagram / calibration curve and compute Brier score.

2. **Interpretability**
   - Use a simple technique (e.g., occlusion or integrated gradients) to highlight which words influence the prediction.

3. **Robustness**
   - Test the model on slightly perturbed text (remove stopwords, shuffle some words) and see how predictions change.

---

**Submission suggestion (if this is coursework):**
- Include your answers to the quick checks.
- Include at least **two** extension tasks with plots + a short discussion.


## References

Chollet, F. (2017). Deep learning with python. Manning Publications.